# SETUP

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
import torch_xla.core.xla_model as xm
import time

# COMPARE JITs

In [2]:
def pyt_function(A, B):
	C = torch.tensor(A.shape, dtype=A.dtype)
	for _ in range(50):
		C = torch.matmul(A, B)
		C = (C + A + B) / 3
		C = C * A * B
		C = C.T
		C = torch.linalg.inv(C)
	d = torch.linalg.det(C)
	return d

def jax_function(A, B):
	C = jnp.array(A.shape, dtype=A.dtype)
	for _ in range(50):
		C = jnp.matmul(A, B)
		C = (C + A + B) / 3
		C = C * A * B
		C = jax.numpy.transpose(C)
		C = jax.numpy.linalg.inv(C)
	d = jax.numpy.linalg.det(C)
	return d

In [3]:
# Input tensors
A_np = np.random.rand(100, 100)
B_np = np.random.rand(100, 100)

A_torch = torch.from_numpy(A_np)
B_torch = torch.from_numpy(B_np)
A_jax = jnp.array(A_np)
B_jax = jnp.array(B_np)

In [4]:
# No JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function(A_torch, B_torch)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT mean time: {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"JAX mean time: {sum(runtimes)/len(runtimes):.6f} seconds")

PyT mean time: 0.014149 seconds
JAX mean time: 0.035055 seconds


In [5]:
# Native JIT (XLA for JAX, TorchScript for PyTorch)

# trace
pyt_function_jit = torch.jit.script(pyt_function)
jax_function_jit = jax.jit(jax_function)
# compile
yt = pyt_function_jit(A_torch, B_torch)
yt = jax_function_jit(A_jax, B_jax).block_until_ready()


runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch, B_torch)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT mean time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JAX mean time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

PyT mean time  (TS): 0.001058 seconds
JAX mean time (XLA): 0.000521 seconds


In [6]:
# JIT + XLA

device = xm.xla_device()
A_torch_xla = A_torch.to(device)
B_torch_xla = B_torch.to(device)

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch_xla, B_torch_xla)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT mean time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"Jax mean time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

PyT mean time (XLA): 0.004996 seconds
Jax mean time (XLA): 0.000303 seconds


# COMPARE AUTOGRADS

In [7]:
# grad
A_torch = A_torch.requires_grad_(True)
B_torch = B_torch.requires_grad_(True)

jax_function_grad = jax.grad(jax_function)

# jit
pyt_function_jit = torch.jit.script(pyt_function)
jax_function_grad_jit = jax.jit(jax_function_grad)

# compile
yt = pyt_function_jit(A_torch, B_torch).backward()
yt = jax_function_grad_jit(A_jax, B_jax).block_until_ready()


In [8]:
# JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch, B_torch).backward()
	end = time.time()
	runtimes.append(end - start)
print(f"PyT mean time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad_jit(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"Jax mean time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

PyT mean time  (TS): 0.001291 seconds
Jax mean time (XLA): 0.001004 seconds


In [9]:
# JIT + XLA

device = xm.xla_device()
A_torch_xla = A_torch.to(device)
B_torch_xla = B_torch.to(device)
yt = pyt_function_jit(A_torch_xla, B_torch_xla).backward()


runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch_xla, B_torch_xla).backward()
	end = time.time()
	runtimes.append(end - start)
print(f"PyT mean time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad_jit(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"Jax mean time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

PyT mean time (XLA): 0.013825 seconds
Jax mean time (XLA): 0.011618 seconds
